# 09 — Batched Qwen/CrafText rollout

Живой synchronous rollout: `B EnvState → B MegaPrompts → one Tunix sampler call → B decode/fallback → vmap(CrafText.step)`. Нужны явные local Qwen weights.


In [ ]:
from pathlib import Path
import jax
from tunix_craftext.batched_rollout import collect_batched_text_rollout, replays_from_batched_rollout
from tunix_craftext.config import load_mvp_config
from tunix_craftext.prompts import MegaPromptRenderer
from tunix_craftext.replay import save_replay
from tunix_craftext.runtime import build_craftext_runtime
from tunix_craftext.tunix_adapter import QwenTunixBackend


In [ ]:
ROOT = next((p for p in (Path.cwd(), *Path.cwd().parents) if (p / 'pyproject.toml').is_file()), None)
if ROOT is None: raise RuntimeError('Run inside tunix-craftext.')
SNAPSHOT = ROOT / 'artifacts/models/qwen25-05b-instruct'
if not SNAPSHOT.is_dir(): raise FileNotFoundError(f'No explicit local snapshot: {SNAPSHOT}')
config = load_mvp_config(ROOT / 'configs/mvp/qwen_craftext.yaml')
runtime = build_craftext_runtime(config)
BATCH_SIZE, HORIZON = 2, 2


In [ ]:
rollout = collect_batched_text_rollout(
    runtime.adapter, MegaPromptRenderer(config.prompt.template),
    QwenTunixBackend(SNAPSHOT, cache_size=2048, seed=config.run.seed),
    actions=runtime.actions, batch_size=BATCH_SIZE, horizon=HORIZON, seed=config.run.seed,
    goal='Stay alive and inspect the world.', max_new_tokens=8,
    invalid_action='fallback', fallback_action_id=runtime.actions.index_of('NOOP'),
)
for step, (decision, resets) in enumerate(zip(rollout.decisions, rollout.reset_after_step, strict=True)):
    print('step', step, 'actions', [d.label for d in decision.actions], 'resets', resets.tolist(), 'rewards', decision.transition.reward.tolist())


In [ ]:
replays = replays_from_batched_rollout(rollout, config_path='configs/mvp/qwen_craftext.yaml', commit='notebook', backend='tunix-single-device:Qwen')
output_dir = ROOT / 'artifacts/trajectories/batched-qwen-rollout'
for index, replay in enumerate(replays):
    path = output_dir / f'env-{index}.json'
    save_replay(path, replay)
    print(path, 'steps=', len(replay.steps), 'tokens=', sum(len(step.token_ids or ()) for step in replay.steps))
